## Radiomics — Extraction & Dataset ML (O3) — Notebook 04

### Objectif
Extraire les features radiomiques des candidats détectés par O2 et construire le dataset ML équilibré (VP/FP).
Ce notebook produit `ml_dataset_balanced.csv`, jeu de données nécessaire à l'entraînement du classifieur (notebook 05).

### Prérequis
- **Librairies** : `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`, `pyradiomics`
- **Données** :
  - `data/processed/PATxxx/volume.npy` + `mask.npy` (produits par O1, notebook 02)
  - `data/processed/PATxxx/candidates.npy` (produits par O2, notebook 03 cellule 5)
- **Modules** : `src.O3_radiomics.*`

### Étapes principales
1. [Imports & configuration](#1-imports--configuration)
2. [Vérification des prérequis](#2-v%C3%A9rification-des-pr%C3%A9requis)
3. [Extraction démo (PAT001)](#3-extraction-d%C3%A9mo-pat001)
4. [Construction du dataset ML complet](#4-construction-du-dataset-ml-complet)
5. [Sélection de features](#5-s%C3%A9lection-de-features)
6. [Visualisation de l'importance des features](#6-visualisation-de-limportance-des-features)

> **Lien avec les notebooks précédents :** Requiert les sorties de `02_preprocessing_demo.ipynb` (O1) et `03_classical_detection_demo.ipynb` (O2).
> **Lien avec le notebook suivant :** `ml_dataset_balanced.csv` est requis par `05_classical_classifier_demo.ipynb`.


## 1. Imports & configuration

### 1.1 Imports et configuration du chemin racine

Import des librairies et des modules de radiomics (`O3_radiomics`).
Définition des chemins vers les données prétraitées.

> **Sortie :** variables `PROCESSED_DIR` et `DATA_DIR` disponibles pour toutes les cellules suivantes.


In [ ]:
# CELLULE 1 - Imports
import sys, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path: sys.path.insert(0, root_path)

from src.O3_radiomics.feature_extractor import EmbolismFeatureExtractor
from src.O3_radiomics.feature_selector  import RadiomicsFeatureSelector
from src.O3_radiomics.build_ml_dataset  import main as build_ml_dataset

sns.set_style('whitegrid')
DATA_DIR      = Path.cwd().parent / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
print('Imports OK')
print(f'data/processed/ existe : {PROCESSED_DIR.exists()}')


## 2. Vérification des prérequis

### 2.1 Contrôle de la présence des fichiers nécessaires

Vérification que tous les fichiers requis sont présents pour chaque patient :
- `volume.npy` (O1) — volume CT prétraité
- `mask.npy` (O1) — masque GT
- `candidates.npy` (O2) — candidats de la détection classique

> **Point d'attention :** si des `candidates.npy` sont manquants, relancer le notebook 03 cellule 5 (batch).


In [ ]:
# CELLULE 2 - Verification prerequisites
patient_dirs = sorted([d for d in PROCESSED_DIR.iterdir()
                       if d.is_dir() and d.name.startswith('PAT')])
n_vol  = sum(1 for d in patient_dirs if (d/'volume.npy').exists())
n_mask = sum(1 for d in patient_dirs if (d/'mask.npy').exists())
n_cand = sum(1 for d in patient_dirs if (d/'candidates.npy').exists())
print(f'Patients trouves   : {len(patient_dirs)}')
print(f'  volume.npy       : {n_vol}/35')
print(f'  mask.npy         : {n_mask}/35')
print(f'  candidates.npy   : {n_cand}/35')
if n_cand < len(patient_dirs):
    print(f'ATTENTION : {len(patient_dirs)-n_cand} patients sans candidates.npy')
    print('  --> Lancer la cellule batch du notebook 03 dabord')
else:
    print('OK : tous les prerequis sont presents')


## 3. Extraction démo (PAT001)

### 3.1 Extraction radiomique sur un patient unique

Démonstration de l'extraction des features radiomiques sur `PAT001`.
Les features extraites incluent des caractéristiques de texture (GLCM, GLRLM), de forme et d'intensité.

> **Durée estimée :** 30–60 secondes par patient.
> **Sortie :** dictionnaire de features radiomiques pour les candidats de PAT001.


In [ ]:
# CELLULE 3 - Demo extraction radiomique sur PAT001
patient_id = 'PAT001'
vol_path  = PROCESSED_DIR / patient_id / 'volume.npy'
mask_path = PROCESSED_DIR / patient_id / 'mask.npy'
assert vol_path.exists() and mask_path.exists()

volume = np.load(vol_path)
masque = np.load(mask_path)
print(f'{patient_id} : {volume.shape}  range [{volume.min():.3f}, {volume.max():.3f}]')
print(f'Voxels embolie : {masque.sum()}')

z_idx   = np.where(masque > 0)[0]
slice_z = int(np.median(z_idx)) if len(z_idx) > 0 else volume.shape[0]//2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(volume[slice_z], cmap='bone'); axes[0].set_title(f'Volume [0,1] — coupe {slice_z}'); axes[0].axis('off')
axes[1].imshow(masque[slice_z], cmap='hot');  axes[1].set_title('Masque embolie (GT)');             axes[1].axis('off')
plt.tight_layout(); plt.show()

extractor = EmbolismFeatureExtractor()
features  = extractor.extract_from_arrays(volume, masque)
print(f'{len(features)} features extraites')
display(pd.DataFrame([features]).T.rename(columns={0:'Valeur'}))


## 4. Construction du dataset ML complet

### 4.1 Construction du dataset ML équilibré (VP + FP)

Extraction des features pour les 35 patients et construction du dataset ML :
- **VP (Vrais Positifs)** : candidats issus des zones annotées GT (embolies confirmées)
- **FP (Faux Positifs)** : candidats O2 hors GT (structures saines détectées à tort)

Le dataset est sauvegardé dans `data/processed/ml_dataset_balanced.csv`.

> **Durée estimée :** 5–15 minutes selon la puissance de la machine.
> **Sortie :** `ml_dataset_balanced.csv` — **requis par le notebook 05**.


In [ ]:
# CELLULE 4 - Construction du dataset ML complet (35 patients)
# Lit volume.npy + mask.npy (GT=VP) + candidates.npy (O2=FP)
# Produit : data/processed/ml_dataset_balanced.csv
CSV_ML = PROCESSED_DIR / 'ml_dataset_balanced.csv'

if CSV_ML.exists():
    df = pd.read_csv(CSV_ML)
    print(f'Dataset existant : {CSV_ML.name}')
    print(f'  Lignes : {len(df)} | VP : {df["has_embolism"].sum()} | FP : {(df["has_embolism"] == 0).sum()}')
    print('  Supprimer le CSV pour regenerer.')
else:
    print('Lancement de build_ml_dataset...')
    build_ml_dataset()
    df = pd.read_csv(CSV_ML)

n_vp = int(df['has_embolism'].sum())
n_fp = len(df) - n_vp
print(f'Dataset : {len(df)} candidats | VP={n_vp} ({100*n_vp/len(df):.1f}%) | FP={n_fp} ({100*n_fp/len(df):.1f}%)')
print(f'Features : {len(df.columns)-3}')
display(df.head(3))

## 5. Sélection de features

### 5.1 Sélection des features les plus discriminantes

Réduction de la dimensionnalité par sélection des features les plus pertinentes pour distinguer VP et FP.
La sélection utilise des critères statistiques (variance, corrélation, importance) pour éliminer les features redondantes ou non informatives.

> **Sortie :** `radiomics_features_selected.csv` + objet `selector` avec les features retenues.


In [ ]:
# CELLULE 5 - Selection de features
from sklearn.model_selection import train_test_split
meta_cols    = ['PatientID', 'candidate_id', 'has_embolism']
feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].fillna(0).replace([float('inf'),float('-inf')], 0)
y = df['has_embolism']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train : {len(X_train)} | Test : {len(X_test)}')

selector = RadiomicsFeatureSelector(variance_threshold=0.01, correlation_threshold=0.90, top_k_features=10)
X_train_sel = selector.select(X_train, y_train)
X_test_sel  = X_test[selector.selected_features]
print(f'Features : {len(feature_cols)} → {len(selector.selected_features)} apres selection')
print(f'Retenues : {selector.selected_features}')

out_csv = PROCESSED_DIR / 'radiomics_features_selected.csv'
df_sel = df[meta_cols + selector.selected_features]
df_sel.to_csv(out_csv, index=False)
print(f'CSV selectionne : {out_csv}')


## 6. Visualisation de l'importance des features

### 6.1 Graphique de l'importance des 10 meilleures features

Affichage des 10 features les plus importantes selon le critère de sélection.
Ce graphique guide le réglage du `correlation_threshold` dans `RadiomicsFeatureSelector`.

> **Sortie :** figure sauvegardée dans `figures/radiomics/feature_importance.png`.


In [ ]:
# CELLULE 6 - Visualisation importance des features
import os
os.makedirs('../figures/radiomics', exist_ok=True)
selector.plot_importance(top_n=10, save_path='../figures/radiomics/feature_importance.png')


### Notes

- **Prerequis** : `volume.npy` + `mask.npy` (O1) + `candidates.npy` (O2 notebook 03 batch)
- **Cellule 4** produit `ml_dataset_balanced.csv` — requis par O4 (`train_classifier.py`)
- **Cellule 5** produit `radiomics_features_selected.csv` — requis par notebook 05
- Dataset naturellement desequilibre (~5-10% VP) → `class_weight='balanced'` dans O4
- Prochaine etape : `04b_batch_extraction_vp_fp.ipynb` (analyse) puis `05_classical_classifier_demo.ipynb`